# CardioRisk — Exploratory Data Analysis

Goal: understand the **combined UCI Heart Disease dataset** (~920 rows from four
hospitals) well enough to justify the modelling and cleaning choices made in
`src/cardiorisk/`. We read the data through the project's own `data.py` so the
notebook and the pipeline never disagree about what 'the data' is.

Run from the repo root inside the project environment:
`jupyter lab notebooks/01_eda.ipynb`

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from cardiorisk.data import load_raw_data
from cardiorisk.preprocessing import CATEGORICAL_FEATURES, NUMERIC_FEATURES
from cardiorisk.target import binarize_target

pd.set_option("display.max_columns", None)
df = load_raw_data("../data/raw/heart_disease.csv")
df["target"] = binarize_target(df["num"])
df.shape

## 1. Shape, sources, and the target

The combined dataset stitches together Cleveland, Hungary, Switzerland, and VA
Long Beach. The original `num` label is severity 0-4; we collapse it to a binary
at-risk flag (`num > 0`).

In [ ]:
print("Rows per source:")
print(df["source"].value_counts())
print("\nBinary target balance:")
print(df["target"].value_counts(normalize=True).round(3))

**Reading:** the dataset is fairly balanced (~55% at-risk), so accuracy is not
wildly misleading — but because false negatives are clinically costly we still
report ROC-AUC, PR-AUC, and a recall-tuned threshold rather than leaning on
accuracy alone.

## 2. Missing values

This is the real reason to use the combined dataset: unlike the 303-row Cleveland
export, several columns have substantial missingness, which motivates the
imputation step inside the pipeline.

In [ ]:
missing = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing[missing > 0]

In [ ]:
miss = missing[missing > 0]
ax = miss.plot.barh(figsize=(7, 4), color="#c0392b")
ax.set_xlabel("% missing")
ax.set_title("Missingness by feature")
ax.invert_yaxis()
plt.tight_layout()

**Reading:** `ca`, `thal`, and `slope` are heavily missing (driven by the
Switzerland/VA cohorts). We treat them as categorical with most-frequent
imputation; `chol` (continuous) is median-imputed. HistGradientBoosting can also
use NaNs natively, which is why it's in the model line-up.

## 3. Numeric features vs. the target

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(4 * len(NUMERIC_FEATURES), 3.5))
for ax, col in zip(axes, NUMERIC_FEATURES, strict=False):
    for label, grp in df.groupby("target"):
        ax.hist(grp[col].dropna(), bins=20, alpha=0.5, label=f"target={label}")
    ax.set_title(col)
axes[0].legend()
plt.tight_layout()

**Reading:** at-risk patients trend toward higher `age` and `oldpeak` and lower
`thalach` (max heart rate) — physiologically sensible and consistent with the
model's learned feature importances.

## 4. Categorical features vs. the target

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), CATEGORICAL_FEATURES, strict=False):
    rate = df.groupby(col)["target"].mean()
    rate.plot.bar(ax=ax, color="#2b6cb0")
    ax.set_title(f"P(at risk) by {col}")
    ax.set_ylim(0, 1)
plt.tight_layout()

**Reading:** asymptomatic chest pain (`cp=4`), exercise-induced angina
(`exang=1`), and a higher number of major vessels (`ca`) are associated with
markedly higher risk — these dominate the SHAP summary in `artifacts/`.

## 5. Correlation among numeric features

In [ ]:
corr = df[[*NUMERIC_FEATURES, "target"]].corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns)
fig.colorbar(im)
ax.set_title("Correlation (numeric)")
plt.tight_layout()

**Reading:** correlations are modest (no near-duplicate features), so there's no
strong multicollinearity to prune. Standardising the numeric features is enough
for the logistic-regression model to behave well.

## Takeaways for modelling
1. Real missingness in `ca`/`thal`/`slope`/`chol` → impute inside the pipeline.
2. Balanced-ish target but asymmetric costs → optimise a recall-tuned threshold.
3. `cp`, `oldpeak`, `exang`, `ca`, `thalach` carry most of the signal — borne out
   by permutation importance and SHAP.